# PPE Compliance Checker — V2 Training (accuracy improvement)

**ITAI 1378 final project.** This notebook implements the accuracy-improvement plan from `docs/dataset_analysis.md`. Every change is matched to a measured property of the dataset:

| Change | Why (measured evidence) |
|---|---|
| imgsz **960** (was 640) | 77–78% of `no_goggle` / `no_boots` / `goggles` boxes are under 1% of image area (median 0.5–0.7%) — at 640px these are ~45-pixel objects. |
| **Oversample** rare-class images | Class imbalance is 20.3:1 (Person 1,790 vs no_boots 88 train instances). |
| **60 epochs, patience 15** (was 40) | Higher resolution + a rebalanced train set shift the plateau; early stopping guards the budget. |
| Evaluate on **val AND test** | V1 was only measured on val (143 imgs); test (141 imgs) is an untouched second measurement. |

**Setup:** Colab → Runtime → Change runtime type → **T4 GPU** → Run all.
First run installs Ultralytics and auto-restarts — when it reconnects, Run all again.
**Budget:** roughly 1.5 h on a free T4 (V1 was 17 min; this trains ~1.6× the images at 2.25× the pixels for up to 1.5× the epochs).


In [ ]:
# Colab preloads torch; installing a package that touches torch needs a runtime
# restart afterward, or `import torch` throws a circular-import error. This cell
# installs Ultralytics once, restarts, then skips on every later run.
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
    print('Installed Ultralytics. Restarting runtime - when it reconnects, do Runtime > Run all again.')
    os.kill(os.getpid(), 9)   # auto-restart so torch imports cleanly
else:
    print('Ultralytics already installed - continuing.')


In [ ]:
import torch, ultralytics
ultralytics.checks()
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Do: Runtime > Change runtime type > T4 GPU, then Run all again.'


## 1 · Configuration

`OVERSAMPLE` is *extra copies* of every train image containing that class (the copy carries all its labels). Factors were chosen from the measured train counts: no_boots 88 instances / 28 images gets ×5 extra; the other violation classes (337–442 instances) get ×2. Val and test are never touched.

In [ ]:
MODEL      = 'yolo11s.pt'   # same family as V1 for a clean comparison; try 'yolo11m.pt' if you have GPU time to spare
IMGSZ      = 960            # V1 used 640 - tiny-box fix
EPOCHS     = 60             # V1 used 40
PATIENCE   = 15
BATCH      = 8              # 960px on a 16 GB T4; drop to 6 if you hit CUDA OOM
OVERSAMPLE = {'no_boots': 5, 'no_goggle': 2, 'no_helmet': 2, 'no_gloves': 2}

# Real V1 reference numbers (from the executed midterm run) for the comparison cell
V1 = {'map50': 0.608, 'map5095': 0.302, 'mp': 0.663, 'mr': 0.585}


## 2 · Download the dataset and build the oversampled train set

We keep the original `train/` untouched and add duplicated copies of rare-class images to `train_os/`. A per-image image is copied `max(factor for the rare classes it contains)` extra times, then a `construction-ppe-v2.yaml` points training at both directories.

In [ ]:
import shutil, glob
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

ds = check_det_dataset('construction-ppe.yaml')   # triggers the 178 MB auto-download on first run
ROOT = Path(ds['path'])
print('Dataset root:', ROOT)

NAMES = {0:'helmet',1:'gloves',2:'vest',3:'boots',4:'goggles',5:'none',6:'Person',
         7:'no_helmet',8:'no_goggle',9:'no_gloves',10:'no_boots'}
ID_BY_NAME = {v:k for k,v in NAMES.items()}
FACTOR = {ID_BY_NAME[k]:v for k,v in OVERSAMPLE.items()}

img_os = ROOT/'images'/'train_os'; lbl_os = ROOT/'labels'/'train_os'
for d in (img_os, lbl_os):
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)

n_src = n_copies = 0
for lf in sorted((ROOT/'labels'/'train').glob('*.txt')):
    classes = {int(l.split()[0]) for l in lf.read_text().splitlines() if l.strip()}
    extra = max([FACTOR.get(c, 0) for c in classes], default=0)
    if extra == 0: continue
    imgs = [p for ext in ('.jpg','.jpeg','.png') for p in [(ROOT/'images'/'train'/(lf.stem+ext))] if p.exists()]
    if not imgs: continue
    img = imgs[0]; n_src += 1
    for k in range(extra):
        shutil.copy(img, img_os/f'{img.stem}_os{k}{img.suffix}')
        shutil.copy(lf,  lbl_os/f'{lf.stem}_os{k}.txt')
        n_copies += 1
print(f'Oversampled {n_src} rare-class images with {n_copies} extra copies.')

yaml_v2 = ROOT/'construction-ppe-v2.yaml'
yaml_v2.write_text(f'''path: {ROOT}
train:
  - images/train
  - images/train_os
val: images/val
test: images/test
names:
''' + ''.join(f'  {k}: {v}\n' for k, v in NAMES.items()))
print('Wrote', yaml_v2)

# show the rebalanced class counts
import collections
cnt = collections.Counter()
for d in ('train','train_os'):
    for lf in (ROOT/'labels'/d).glob('*.txt'):
        for l in lf.read_text().splitlines():
            if l.strip(): cnt[int(l.split()[0])] += 1
print('\nEffective train instances after oversampling:')
for c in sorted(cnt, key=cnt.get, reverse=True):
    print(f'  {NAMES[c]:12s} {cnt[c]:5d}')


## 3 · Train

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(yaml_v2),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    patience=PATIENCE, cos_lr=True, name='ppe_v2', plots=True,
)
RUN_DIR = str(results.save_dir)
print('Training outputs saved to:', RUN_DIR)


## 4 · Evaluate — val AND test, with V1 comparison

In [ ]:
def report(split):
    m = model.val(data=str(yaml_v2), split=split, imgsz=IMGSZ)
    print(f'\n=== {split.upper()} ===')
    print(f'mAP@50    : {m.box.map50:.3f}')
    print(f'mAP@50-95 : {m.box.map:.3f}')
    print(f'precision : {m.box.mp:.3f}   recall: {m.box.mr:.3f}')
    return m

m_val  = report('val')
m_test = report('test')

print('\n=== V1 (640px, 40 epochs, no oversampling) vs V2 - validation split ===')
print(f"{'metric':12s} {'V1':>8s} {'V2':>8s} {'delta':>8s}")
for k, attr in [('mAP@50','map50'), ('mAP@50-95','map'), ('precision','mp'), ('recall','mr')]:
    v1 = V1[{'map50':'map50','map':'map5095','mp':'mp','mr':'mr'}[attr]]
    v2 = float(getattr(m_val.box, attr))
    print(f'{k:12s} {v1:8.3f} {v2:8.3f} {v2-v1:+8.3f}')
print('\nNOTE: no_boots has only 4 val / 23 test instances - treat its per-class numbers as noise (see docs/dataset_analysis.md).')


## 5 · Compliance demo — positive-evidence rule

`docs/dataset_analysis.md`, Finding 2: the `no_*` classes are trained on off-domain stock photos, so the V1 rule (*flag only if a `no_*` box fires or a vest is missing*) misses real violations — workers without hard hats passed as COMPLIANT.

The V2 rule requires **positive detection of required gear** (helmet + vest — the model's 0.80+ mAP classes) and uses `no_*` hits as additional negative evidence. Same logic ships in `src/predict_compliance.py`.

In [ ]:
import cv2, glob, os
from IPython.display import Image as IPyImage, display

best = os.path.join(RUN_DIR, 'weights', 'best.pt')
det = YOLO(best)
names = det.names
REQUIRED = {'helmet', 'vest'}          # positive gear that must be detected on each person
VIOLATION_LABELS = {'no_helmet', 'no_goggle'}

def center_in(box, person):
    cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
    return person[0] <= cx <= person[2] and person[1] <= cy <= person[3]

def annotate(img_path, conf=0.35):
    img = cv2.imread(img_path)
    r = det(img_path, conf=conf, imgsz=IMGSZ, verbose=False)[0]
    dets = [(names[int(c)], list(map(float, b))) for c, b in zip(r.boxes.cls, r.boxes.xyxy)]
    persons = [b for n, b in dets if n.lower() == 'person']
    others  = [(n, b) for n, b in dets if n.lower() != 'person']
    if not persons:
        persons = [[0, 0, img.shape[1], img.shape[0]]]
    for p in persons:
        near = [n for n, b in others if center_in(b, p)]
        missing    = [g for g in REQUIRED if g not in near]      # positive evidence
        violations = [n for n in near if n in VIOLATION_LABELS]  # negative evidence
        problems = missing + [v.replace('no_', 'no ') for v in violations]
        ok = not problems
        color = (0, 170, 0) if ok else (0, 0, 220)
        label = 'COMPLIANT' if ok else 'NON-COMPLIANT: missing ' + ', '.join(problems)
        x1, y1, x2, y2 = map(int, p)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
        cv2.putText(img, label, (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    return img

OUT = '/content/compliance_out_v2'; os.makedirs(OUT, exist_ok=True)
val_imgs = sorted(glob.glob(str(ROOT/'images'/'val'/'*')))[:8]
for p in val_imgs:
    out = os.path.join(OUT, os.path.basename(p))
    cv2.imwrite(out, annotate(p))
    display(IPyImage(filename=out, width=520))
print('Saved to', OUT)


## 6 · Zip everything for download

In [ ]:
import shutil, os
os.makedirs('/content/ppe_v2_results', exist_ok=True)
shutil.copy(os.path.join(RUN_DIR, 'weights', 'best.pt'), '/content/ppe_v2_results/best_v2.pt')
for f in ['results.png','results.csv','confusion_matrix.png','confusion_matrix_normalized.png','PR_curve.png','val_batch0_pred.jpg']:
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p): shutil.copy(p, '/content/ppe_v2_results/')
shutil.copytree('/content/compliance_out_v2', '/content/ppe_v2_results/compliance_out_v2', dirs_exist_ok=True)
shutil.make_archive('/content/ppe_v2_results', 'zip', '/content/ppe_v2_results')
print('Zipped: /content/ppe_v2_results.zip - download it, and drop the contents into results/ in the repo.')
try:
    from google.colab import files
    files.download('/content/ppe_v2_results.zip')
except Exception as e:
    print('(manual download from the Files panel)', e)


## Outputs

- `best_v2.pt` — the improved model
- `results.png`, `results.csv`, confusion matrices, PR curve — V2 metric artifacts
- `compliance_out_v2/` — demo outputs under the positive-evidence rule
- The printed **V1 vs V2 comparison table** — these are the numbers that go into the final slides and README (never estimates)
